In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raheenz/ubl-news-price/UBL.csv
/kaggle/input/datasets/raheenz/ubl-news-price/ubl_news_clean.csv


In [3]:
import os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ---- CONFIG (Kaggle) ----
NEWS_PATH  = "/kaggle/input/datasets/raheenz/ubl-news-price/ubl_news_clean.csv"
PRICE_PATH = "/kaggle/input/datasets/raheenz/ubl-news-price/UBL.csv"
FEAT_DIR   = "/kaggle/working"
START      = "2021-09-30"
MODEL_TYPE = "mlp"    
RERUN_FINBERT = False
HALF_LIFE  = 3
N_PCA      = 8
os.makedirs(FEAT_DIR, exist_ok=True)

print("NEWS exists:", os.path.exists(NEWS_PATH))
print("PRICE exists:", os.path.exists(PRICE_PATH))

NEWS exists: True
PRICE exists: True


In [4]:
# ============ STEP 1: FinBERT on the news (cached) ============
CACHE_S, CACHE_E = f"{FEAT_DIR}/article_scored.csv", f"{FEAT_DIR}/article_emb.npy"

if os.path.exists(CACHE_S) and os.path.exists(CACHE_E) and not RERUN_FINBERT:
    arts = pd.read_csv(CACHE_S, parse_dates=["pub_date", "feature_date"]); E = np.load(CACHE_E)
    print(f"loaded cached FinBERT output: {len(arts)} articles")
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    arts = pd.read_csv(NEWS_PATH, parse_dates=["pub_date", "feature_date"])
    tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device).eval()
    id2 = {int(k): v.lower() for k, v in model.config.id2label.items()}
    POS, NEG, NEU = [[k for k, v in id2.items() if v == name][0] for name in ("positive", "negative", "neutral")]

    # split long articles into 510-token chunks
    chunks = []
    for i, (t, x) in enumerate(zip(arts["title"], arts["text"])):
        ids = tok(f"{t}. {x}", add_special_tokens=False)["input_ids"]
        for s in range(0, max(len(ids), 1), 510):
            chunks.append((i, ids[s:s + 510]))
    print(f"{len(arts)} articles -> {len(chunks)} chunks")

    BATCH, probs, embs = 8, [], []
    with torch.no_grad():
        for b in range(0, len(chunks), BATCH):
            batch = [[tok.cls_token_id] + ids + [tok.sep_token_id] for _, ids in chunks[b:b + BATCH]]
            L = max(len(x) for x in batch)
            input_ids = torch.tensor([x + [tok.pad_token_id] * (L - len(x)) for x in batch]).to(device)
            out = model(input_ids=input_ids, attention_mask=(input_ids != tok.pad_token_id).long(), output_hidden_states=True)
            probs.append(torch.softmax(out.logits, -1).cpu().numpy())
            embs.append(out.hidden_states[-1][:, 0, :].cpu().numpy())
            if (b // BATCH) % 10 == 0: print(f"  {min(b + BATCH, len(chunks))}/{len(chunks)} chunks")
    probs, embs = np.vstack(probs), np.vstack(embs)

    # chunks -> articles
    rows = np.array([r for r, _ in chunks]); w = np.array([len(ids) for _, ids in chunks], float)
    P, E, W = np.zeros((len(arts), 3)), np.zeros((len(arts), 768)), np.zeros(len(arts))
    np.add.at(P, rows, probs * w[:, None]); np.add.at(E, rows, embs * w[:, None]); np.add.at(W, rows, w)
    P /= W[:, None]; E /= W[:, None]
    arts["p_pos"], arts["p_neg"], arts["p_neu"] = P[:, POS], P[:, NEG], P[:, NEU]
    arts["sentiment"] = arts["p_pos"] - arts["p_neg"]
    arts.to_csv(CACHE_S, index=False); np.save(CACHE_E, E)
    print("saved FinBERT output to", FEAT_DIR)

print(arts["sentiment"].describe().round(3))
print("\nMost positive:"); print(arts.nlargest(5, "sentiment")[["pub_date", "title"]].to_string(index=False))
print("\nMost negative:"); print(arts.nsmallest(5, "sentiment")[["pub_date", "title"]].to_string(index=False))

device: cuda


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (700 > 512). Running this sequence through the model will result in indexing errors


277 articles -> 475 chunks
  8/475 chunks
  88/475 chunks
  168/475 chunks
  248/475 chunks
  328/475 chunks
  408/475 chunks
saved FinBERT output to /kaggle/working
count    277.000
mean       0.229
std        0.568
min       -0.966
25%       -0.032
50%        0.256
75%        0.774
max        0.939
Name: sentiment, dtype: float64

Most positive:
  pub_date                                                                    title
2021-07-16      Analyst Briefing: PTCL plans CAPEX of Rs20bn to expand FTHH Network
2026-05-14                                JazzWorld delivers strong Q1 2026 results
2025-10-15 UBL profit soars past Rs. 100 billion, declares Rs. 21.50 total dividend
2026-01-19                JazzWorld, UBL seal Pakistan’s largest interest rate swap
2025-10-15                      UBL profit jumps 89% YoY to Rs35.36bn in Q3 of 2025

Most negative:
  pub_date                                         title
2026-04-14      PSX sees a sharp and broad-based decline
2026-03-28      

In [5]:
# ============ STEP 2: price data + daily news features ============
px = pd.read_csv(PRICE_PATH)
px.columns = [c.strip().upper() for c in px.columns]
px["DATE"] = pd.to_datetime(px["DATE"])
px = px[px["DATE"] >= START].drop_duplicates()
ok = ((px[["OPEN", "HIGH", "LOW", "CLOSE"]] > 0).all(axis=1) & (px["VOLUME"] >= 0)
      & (px["HIGH"] >= px[["OPEN", "CLOSE", "LOW"]].max(axis=1)) & (px["LOW"] <= px[["OPEN", "CLOSE"]].min(axis=1)))
px = px[ok].drop_duplicates("DATE").sort_values("DATE").reset_index(drop=True)

# each article -> the first trading day on/after its feature_date
arts["row"] = np.arange(len(arts))
a = pd.merge_asof(arts.sort_values("feature_date"), px[["DATE"]].rename(columns={"DATE": "trade_date"}),
                  left_on="feature_date", right_on="trade_date", direction="forward").dropna(subset=["trade_date"]).reset_index(drop=True)
E_s = E[a["row"].values]

g = a.groupby("trade_date")
daily = pd.DataFrame({"n_articles": g.size(), "sent_mean": g["sentiment"].mean(),
                      "sent_extreme": g["sentiment"].agg(lambda s: s.loc[s.abs().idxmax()])})
df = px.set_index("DATE").join(daily)
df["has_news"] = df["n_articles"].notna().astype(int)
df[["n_articles", "sent_mean", "sent_extreme"]] = df[["n_articles", "sent_mean", "sent_extreme"]].fillna(0)

D = np.zeros((len(df), 768)); pos = {d: i for i, d in enumerate(df.index)}
for d, idx in g.indices.items(): D[pos[d]] = E_s[idx].mean(axis=0)

# news recency / memory features
ds, last, m, memo, decay = [], -1, 0.0, [], 0.5 ** (1 / HALF_LIFE)
for i, (h, s) in enumerate(zip(df["has_news"], df["sent_mean"])):
    if h: last = i
    ds.append(i - last if last >= 0 else 60); m = m * decay + s; memo.append(m)
df["recency"] = 0.5 ** (np.minimum(ds, 60) / HALF_LIFE)
df["sent_decay"] = memo

# ---- TARGET: raw CLOSE (to match friend's setup) ----
df["target"] = df["CLOSE"]

# ---- FEATURES: news only (no technical) ----
NEWS = ["n_articles", "sent_mean", "sent_extreme", "has_news", "recency", "sent_decay"]
keep = df[["target"]].notna().all(axis=1).values
df, D = df[keep].copy(), D[keep]
print(f"{len(df)} usable days | with news: {int(df.has_news.sum())} ({df.has_news.mean():.1%})")

1176 usable days | with news: 167 (14.2%)


In [6]:
# ============ STEP 3: train MLP — news-only, predict raw CLOSE ============
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

tf.random.set_seed(42)

n = len(df); tr_end, va_end = int(n * 0.70), int(n * 0.85)
tr, va, te = slice(0, tr_end), slice(tr_end, va_end), slice(va_end, n)
print("train:", df.index[0].date(), "->", df.index[tr_end - 1].date(),
      "| val ->", df.index[va_end - 1].date(), "| test:", df.index[va_end].date(), "->", df.index[-1].date())

# FinBERT embedding -> PCA (fit on train news days only; zero on no-news days)
has = df["has_news"].values
pca = PCA(N_PCA, random_state=0).fit(D[tr][has[tr] == 1])
EMB = [f"pc{i}" for i in range(N_PCA)]
df[EMB] = pca.transform(D) * has[:, None]

# ---- NEWS-ONLY features  ----
FEATURES = NEWS + EMB
print("Features used:", len(FEATURES), FEATURES)

# ---- scale X (fit on train only) ----
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(df[FEATURES].values[tr])
X_val   = scaler_X.transform(df[FEATURES].values[va])
X_test  = scaler_X.transform(df[FEATURES].values[te])
X_all   = scaler_X.transform(df[FEATURES].values)

# ---- scale y (raw CLOSE, fit on train only) ----
y = df["target"].values
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y[tr].reshape(-1, 1))
y_val   = scaler_y.transform(y[va].reshape(-1, 1))
y_test  = scaler_y.transform(y[te].reshape(-1, 1))

# ---- Keras MLP — same architecture as friend's ----
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1)
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# ---- predict ----
y_pred_scaled = model.predict(X_all, verbose=0)
pred = scaler_y.inverse_transform(y_pred_scaled).flatten()

df["pred_close"] = pred

# ---- evaluate on test set ----
t = df.iloc[te]
y_actual = t["target"].values
y_pred   = t["pred_close"].values

mae  = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
r2   = r2_score(y_actual, y_pred)

print("\n===== TEST RESULTS (news-only MLP, raw CLOSE) =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")

out = df.iloc[te][["CLOSE", "target", "pred_close", "has_news"]].round(3)
out.index.name = "date"
out.to_csv(f"{FEAT_DIR}/predictions_test.csv")
print("\nLast 15 test days:")
print(out.tail(15).to_string())

train: 2021-09-30 -> 2025-01-23 | val -> 2025-10-10 | test: 2025-10-13 -> 2026-06-30
Features used: 14 ['n_articles', 'sent_mean', 'sent_extreme', 'has_news', 'recency', 'sent_decay', 'pc0', 'pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6', 'pc7']


I0000 00:00:1792004422.185105      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12804 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792004422.189905      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,073 (12.00 KB)

 Trainable params: 3,073 (12.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
25/26 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0127 - mae: 0.7345

I0000 00:00:1792004427.182007     260 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - loss: 0.9968 - mae: 0.7540 - val_loss: 11.9184 - val_mae: 3.3156
Epoch 2/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8877 - mae: 0.7324 - val_loss: 12.1291 - val_mae: 3.3425
Epoch 3/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8402 - mae: 0.7083 - val_loss: 12.0038 - val_mae: 3.3211
Epoch 4/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8196 - mae: 0.7000 - val_loss: 12.0608 - val_mae: 3.3282
Epoch 5/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8031 - mae: 0.6917 - val_loss: 12.0268 - val_mae: 3.3225
Epoch 6/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7897 - mae: 0.6861 - val_loss: 12.0260 - val_mae: 3.3213
Epoch 7/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7772 - mae: 0.6808 - val_loss: 12.0298 - val_mae: 3.3210
Epoch 8/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7650 - mae: 0.6750 - val_loss: 12.0061 - val_mae: 3.3168
Epoch 9/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7545 -

In [7]:
# ============ B: predict NEXT-DAY LOG RETURN  ============
import tensorflow as tf
from tensorflow.keras import Sequential, layers, callbacks
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error as MSE, r2_score

tf.random.set_seed(42); np.random.seed(42)

# ---- TARGET = next-day log return ----
df["target_ret"] = np.log(df["CLOSE"]).diff().shift(-1)
# df = df.dropna(subset=["target_ret"]).reset_index(drop=True)
mask = df["target_ret"].notna().values
df = df[mask].reset_index(drop=True)
D  = D[mask]       
has = df["has_news"].values
n = len(df); tr_end, va_end = int(n * 0.70), int(n * 0.85)
tr, va, te = slice(0, tr_end), slice(tr_end, va_end), slice(va_end, n)

pca = PCA(N_PCA, random_state=0).fit(D[tr][has[tr] == 1])
EMB = [f"pc{i}" for i in range(N_PCA)]
df[EMB] = pca.transform(D) * has[:, None]

FEATURES = NEWS + EMB

# ---- scale X ----
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(df[FEATURES].values[tr])
X_val   = scaler_X.transform(df[FEATURES].values[va])
X_all   = scaler_X.transform(df[FEATURES].values)

# ---- scale y (return) ----
y = df["target_ret"].values
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y[tr].reshape(-1, 1))
y_val   = scaler_y.transform(y[va].reshape(-1, 1))

# ---- MLP ----
model = Sequential([
    layers.Input((X_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])
model.compile("adam", "mse", metrics=["mae"])
model.fit(X_train, y_train,
          validation_data=(X_val, y_val),
          epochs=200, batch_size=32,
          callbacks=[callbacks.EarlyStopping(patience=20, restore_best_weights=True)],
          verbose=1)

# ---- predict returns ----
pred_ret_scaled = model.predict(X_all, verbose=0)
pred_ret = scaler_y.inverse_transform(pred_ret_scaled).ravel()

# ---- convert to price ----
c0 = df["CLOSE"].values                 # today's close
pc = c0[te]                             # prev close on test
pred_next_close = c0 * np.exp(pred_ret) # implied predicted close
actual_next_close = c0 * np.exp(y)      # actual next close (via actual return)

nc = actual_next_close[te]
pred = pred_next_close[te]

# ---- evaluate ----
mae  = MAE(nc, pred)
rmse = np.sqrt(MSE(nc, pred))
mape = np.mean(np.abs((nc - pred) / nc)) * 100
r2   = r2_score(nc, pred)
da   = np.mean(np.sign(nc - pc) == np.sign(pred - pc)) * 100   # direction accuracy

print("\n===== TEST RESULTS (news-only MLP, NEXT-DAY RETURN) =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")
print(f"DirAcc: {da:.1f}%")

out = df.iloc[te][["CLOSE"]].copy()
out["actual_next_close"] = nc
out["pred_next_close"]   = pred
out["target_ret"]        = y[te]
out["pred_ret"]          = pred_ret[te]
out["has_news"]          = df.iloc[te]["has_news"].values
out.index.name = "date"
out.round(4).to_csv(f"{FEAT_DIR}/predictions_return.csv")
print("\nSaved predictions_return.csv")
print(out.tail(10).to_string())

Epoch 1/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - loss: 1.0873 - mae: 0.7278 - val_loss: 10.7799 - val_mae: 1.0919
Epoch 2/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.0057 - mae: 0.6923 - val_loss: 10.7630 - val_mae: 1.0833
Epoch 3/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9867 - mae: 0.6827 - val_loss: 10.7600 - val_mae: 1.0861
Epoch 4/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9778 - mae: 0.6793 - val_loss: 10.7574 - val_mae: 1.0872
Epoch 5/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9703 - mae: 0.6760 - val_loss: 10.7562 - val_mae: 1.0876
Epoch 6/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9639 - mae: 0.6730 - val_loss: 10.7542 - val_mae: 1.0881
Epoch 7/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9583 - mae: 0.6707 - val_loss: 10.7523 - val_mae: 1.0896
Epoch 8/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9536 - mae: 0.6687 - val_loss: 10.7476 - val_mae: 1.0902
Epoch 9/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - lo